# このノートだけ

開くのはこのページだけです。ほかのノートは開かない。

表は動画になりません。動画は一番下の **焼く** です。投稿しません。

## 押す順番

ランタイムは最初から **GPU** にします。途中で変えない。変えると、ここまでのファイルが消えます。

1. **読み込み**
2. **選ぶ** で、やりたいことを1つ選んで実行
3. **実行**
4. **焼く**（ready のあと。焼くにチェックを入れて押す。残りの範囲を続けて焼いて、そろったら1本につなぐ。mp4 はマイドライブの affi-bake に残る）

見た目を変えるときだけ、2と3のあいだに、話の名前が同じ見た目のセルを1つ実行します。変えないときは飛ばします。初期値です。

上のメニューの「すべてのセルを実行」は押しません。焼くはオフのままなので、全部実行しても動画は始まりません。

## 選び方

**表を見る**

美容、ドッグフード、見守りカメラ、婚活の作り方の表が出ます。ジョブは書きません。

**話でジョブを書く**

話を1つ選びます。ドッグフードだけ、インタビュー・咀嚼・ダンス・会話も選びます。

中身を1つ選びます。テーマは書きません。台詞は入っています。

- **テンプレ** … 用意してある短い場面と台詞
- **オマージュ** … 同じ絵とカメラの順で、別の短い台詞
- **元の型のまま** … 型の絵とカメラの順。短い新しい台詞。元の顔、元の台詞、曲名は入りません

静止画は空でよいです。空なら文章から焼きます。ファイルを書くと、その画像が最初のコマになります。台詞を変えたいときだけ、台詞の欄に1行ずつ書きます。

全部の範囲が焼けたら、型の秒に切ってつなぎます。字幕は型が焼く話だけ、型に書いてある位置と色で焼きます。自分の曲は「曲」にファイルを書くと、つないだあとに重ねます。美容だけ、題字とロゴも書けます。

**自分の文で1本**

プロンプトを自分で書きます。秒と画面も「選ぶ」で選びます。静止画が空なら T2V、ファイルがあれば手入力で I2V です。「実行」が ready と出たら、一番下の「焼く」です。

**元動画を再現**

元の mp4 の場所を「元動画」に書きます。Ref2VA がそのファイルを参照にします。カメラ、カット、声、曲はファイルが持ちます。
H3 は1本 5〜14.4 秒なので、長い動画は範囲に分かれます。範囲の境は、カットと無音の位置に置き、せりふの途中では切りません。画質優先は短く切って画角を上げます。
参照の範囲は 24 コマ、音は PCM にそろえます。生成が範囲より長い分は、止め絵と無音にしてあとで切ります。5 秒未満の動画も、この形で 5 秒にしてから元の秒に戻します。
測ったカットだけを、その時刻でプロンプトに書きます。音声から語の時刻まで読めたせりふを、聞こえるショットに入れます。聞き取りにくい語は [unclear] です。読めないときは口が参照の音声に合わせ、文は足しません。
HF_TOKEN があり、pyannote/speaker-diarization-3.1 と pyannote/segmentation-3.0 の利用条件に同意していれば、話者を S1、S2 と分けます。埋め込みは WeSpeaker の VoxCeleb ResNet34_LM（PyTorch）です。公式の .onnx は使いません。入らなければ1人として並べます。楽器が判定できたときだけ、曲の行に楽器名を足します。
FL2VA の Turbo は載せません。重みは transformer_ref です。静止画は空のままが、元の見た目です。
焼いたあと、範囲ごとに音のずれ、カット、せりふの文字を元動画と比べて出します。口の動きそのものは測りません。

## 4つの話

1. **美容：材料のキャラ**（the.care.logic）
   材料のキャラが1体で悩み、対処して、笑顔で終わる。30秒を10秒ずつ3つ。見た目を変えるときは材料・場所・口調。

2. **ドッグフード：犬**（nuts0629）
   同じ犬。インタビューは人がマイクを向ける8秒。咀嚼は8秒を4回。ダンスは全身で10秒。会話は吹き出し。見た目を変えるときは犬・場所・口調。型を選ぶのはこの話だけ。

3. **見守り：猫2匹の言い合い**（junjun_ranran）
   猫が2匹で言い合い、最後に人が少し出る。45秒。見守りカメラの映像ではない。見た目を変えるときは猫2匹・人・場所・口調。

4. **婚活：男女の会話**（yako.shiawasekon）
   成人の男女が会話して、最後に関係が変わる。60秒。見た目を変えるときは女性・男性・場所・口調。

一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や、元のアカウントの人・動物に似せる文は、そこで止まります。

秒数・カット・字幕は型のままです。会話（ドッグフード）は、型に区間が書いてある 0〜19 秒だけを作ります。

Checkpoint は MiniMax-H3。話のジョブと手入力は、FL2VA の Turbo と、動作のつながりを使います。
元動画の再現は Ref2VA です。同じ Turbo は載せません。


In [ ]:
#@title 読み込み { display-mode: "form" }
#@markdown 最初にこのセルを実行する。調査日は 2026-10-07。初めてなら変えない。
DATE = "2026-10-07"  #@param {type:"string"}

from pathlib import Path
import sys
import urllib.request

BRANCH = "cursor/affi-template-bake-44d6"
REPO = "fireworker011/Research"
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/research/affi-templates"
STRONG_MIN = 10
COMPARE_MIN = 3
THIN_BELOW = 10
MIN_GAP_PT = 10
LOCAL = Path("research/affi-templates")
if not (LOCAL / "affi_bake.py").is_file():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "research/affi-templates/affi_bake.py").is_file():
            LOCAL = candidate / "research/affi-templates"
            break
if not (LOCAL / "affi_bake.py").is_file():
    LOCAL = Path("/content/affi-templates") if Path("/content").is_dir() else Path(".affi-templates-download")
    files = [
        "affi_genre_templates.py",
        "affi_reference.py",
        "affi_av.py",
        "affi_media.py",
        "affi_speech.py",
        "affi_speaker.py",
        "affi_finish.py",
        "affi_match.py",
        "affi_bake.py",
        "reference-accounts/hypotheses.yaml",
        "reference-accounts/results.csv",
        "reference-accounts/looks.yaml",
        "reference-accounts/templates/the.care.logic.yaml",
        "reference-accounts/templates/nuts0629.yaml",
        "reference-accounts/templates/junjun_ranran.yaml",
        "reference-accounts/templates/yako.shiawasekon.yaml",
        f"data/{DATE}/accounts.csv",
    ]
    for rel in files:
        dest = LOCAL / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{rel}", dest)
        print("取りました", rel)
    try:
        urllib.request.urlretrieve(f"{RAW}/data/{DATE}/videos.csv", LOCAL / "data" / DATE / "videos.csv")
    except Exception:
        print("動画一覧は無し。アカウント一覧だけで計算する。")
sys.path.insert(0, str(LOCAL))
import affi_reference as ref
ref.ROOT = LOCAL
ref.TEMPLATES = LOCAL / "reference-accounts" / "templates"
ref.HYPOTHESES = LOCAL / "reference-accounts" / "hypotheses.yaml"
ref.RESULTS = LOCAL / "reference-accounts" / "results.csv"
ref.LOOKS = LOCAL / "reference-accounts" / "looks.yaml"
DATA_DIR = LOCAL / "data" / DATE
TABLE_OUT = LOCAL / "templates"
print("読みました", LOCAL)
print("次は「選ぶ」を実行してください。")


In [ ]:
#@title 選ぶ { display-mode: "form" }
#@markdown やりたいことを1つ選んで、このセルを実行する。上の「すべてのセルを実行」は押さない。
何をする = "話でジョブを書く" #@param ["表を見る", "話でジョブを書く", "自分の文で1本", "元動画を再現"]
#@markdown 話でジョブを書くときだけ、下の話・型・中身・台詞を使う。表を見る、自分の文、元動画を再現では無視する。
話 = "ドッグフード：犬（nuts0629）" #@param ["美容：材料のキャラ（the.care.logic）", "ドッグフード：犬（nuts0629）", "見守り：猫2匹の言い合い（junjun_ranran）", "婚活：男女の会話（yako.shiawasekon）"]
#@markdown ドッグフードだけ下を使う。インタビューは人がマイクを向ける8秒。咀嚼は8秒を4回。ダンスは全身で10秒。会話は吹き出し。ほかの話では無視する。
ドッグフードの型 = "インタビュー（既定）" #@param ["インタビュー（既定）", "咀嚼", "ダンス", "会話"]
#@markdown 中身。テンプレは用意した短い場面。オマージュは同じ絵の順で、別の短い台詞。元の型のままは型の絵とカメラ。元の顔、元の台詞、曲名は入らない。
中身 = "テンプレ" #@param ["テンプレ", "オマージュ", "元の型のまま"]
#@markdown 台詞を置き換えるときだけ書く。空なら、中身の台詞を使う。1行が1カット。
台詞 = "" #@param {type:"raw"}
#@markdown 静止画は空でよい。話と自分の文では、空なら文章から焼く。ファイルを書くと、その画像が最初のコマになる。元動画を再現では、空のままが元の見た目。ファイルを書くと、見た目だけその画像になる。
静止画 = "" #@param {type:"string"}
#@markdown 元動画を再現するときだけ、mp4 の場所を書く。空なら動かない。カメラ、カット、声、曲は、そのファイルが持つ。読めたせりふは口がその文だけを作る。FL2VA の Turbo は載せない。
元動画 = "" #@param {type:"string"}
#@markdown 元動画の範囲の切り方。画質優先は短く切って画角を上げる。本数は増える。本数を減らすは長く切る。画角は下がる。
範囲の切り方 = "画質優先（短く切る）" #@param ["画質優先（短く切る）", "本数を減らす（長く切る）"]
#@markdown 話でジョブを書くときだけ。自分の曲を重ねるときは、その音声ファイルの場所。空なら曲は型のまま。
曲 = "" #@param {type:"string"}
#@markdown 美容だけ。題字は最初の 0.07 秒の1枚絵。ロゴは右上の小さな白文字で、自分の名前。空なら入れない。
題字 = "" #@param {type:"string"}
ロゴ = "" #@param {type:"string"}
#@markdown 自分の文で1本のときだけ、下を使う。静止画が空なら T2V。ファイルがあれば I2V。
秒 = "10" #@param ["5", "6", "8", "10", "12", "14"]
画面 = "9:16" #@param ["9:16", "16:9"]
プロンプト = "" #@param {type:"raw"}

import affi_bake

HANDLE, MODE = affi_bake.resolve_account(話, ドッグフードの型)
print(affi_bake.next_step(何をする))
if 何をする == "話でジョブを書く":
    print(affi_bake.describe_account(HANDLE, MODE))


# 見た目を変えるときだけ

「選ぶ」が **話でジョブを書く** のときだけ使います。見出しが今の話と同じセルを1つ実行します。

迷ったら、この4つは実行しません。初期値を使います。初期値は元のアカウントの顔ではありません。

ほかの話のセルを実行しても、見た目は入りません。


In [ ]:
#@title 美容：材料のキャラの見た目 { display-mode: "form" }
#@markdown 材料のキャラが1体で悩み、対処して、笑顔で終わる。30秒を10秒ずつ3つ。迷ったら初期値のまま。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 材料のキャラ
材料 = "茶葉" #@param ["茶葉", "石けん", "タオル", "マグカップ", "その他（直接入力）"] {allow-input: true}
作り = "粘土" #@param ["粘土", "折り紙", "水彩", "シンプルな3D", "その他（直接入力）"] {allow-input: true}
キャラの色 = "生成り" #@param ["生成り", "ミント", "紺", "黄土", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "リビング" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "北欧" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "白と木" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "昼の自然光" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "the.care.logic")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "材料": 材料,
            "作り": 作り,
            "キャラの色": キャラの色,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


In [ ]:
#@title ドッグフード：犬の見た目 { display-mode: "form" }
#@markdown 同じ犬。インタビューは人がマイクを向ける8秒。咀嚼は8秒を4回。ダンスは全身で10秒。会話は吹き出し。人の顔はここでは選びません。迷ったら初期値のまま。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 犬（会話の2匹もこの見た目）
犬の種類 = "犬" #@param ["犬", "猫", "うさぎ", "ハムスター", "鳥", "その他（直接入力）"] {allow-input: true}
犬種 = "柴" #@param ["柴", "プードル", "ビーグル", "マンチカン", "スコティッシュフォールド", "その他（直接入力）"] {allow-input: true}
犬の毛 = "黒の短毛" #@param ["黒の短毛", "白の短毛", "茶色のぶち", "グレー一色", "クリームの短毛", "その他（直接入力）"] {allow-input: true}
犬の体 = "小柄" #@param ["小柄", "普通", "がっしり", "脚が長め", "その他（直接入力）"] {allow-input: true}
犬の衣装 = "無地の首輪" #@param ["無地の首輪", "ニット", "何も着けない", "バンダナ", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "リビング" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "北欧" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "白と木" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "昼の自然光" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "nuts0629")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "犬の種類": 犬の種類,
            "犬種": 犬種,
            "犬の毛": 犬の毛,
            "犬の体": 犬の体,
            "犬の衣装": 犬の衣装,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


In [ ]:
#@title 見守り：猫2匹の見た目 { display-mode: "form" }
#@markdown 猫が2匹で言い合い、最後に人が少し出る。45秒。見守りカメラの映像ではない。迷ったら初期値のまま（猫が2匹）。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 1匹目の猫
一匹目の種類 = "猫" #@param ["犬", "猫", "うさぎ", "ハムスター", "鳥", "その他（直接入力）"] {allow-input: true}
一匹目の猫種 = "マンチカン" #@param ["柴", "プードル", "ビーグル", "マンチカン", "スコティッシュフォールド", "その他（直接入力）"] {allow-input: true}
一匹目の毛 = "グレー一色" #@param ["黒の短毛", "白の短毛", "茶色のぶち", "グレー一色", "クリームの短毛", "その他（直接入力）"] {allow-input: true}
一匹目の体 = "普通" #@param ["小柄", "普通", "がっしり", "脚が長め", "その他（直接入力）"] {allow-input: true}
一匹目の衣装 = "何も着けない" #@param ["無地の首輪", "ニット", "何も着けない", "バンダナ", "その他（直接入力）"] {allow-input: true}

#@markdown 2匹目の猫
二匹目の種類 = "猫" #@param ["犬", "猫", "うさぎ", "ハムスター", "鳥", "その他（直接入力）"] {allow-input: true}
二匹目の猫種 = "スコティッシュフォールド" #@param ["柴", "プードル", "ビーグル", "マンチカン", "スコティッシュフォールド", "その他（直接入力）"] {allow-input: true}
二匹目の毛 = "白の短毛" #@param ["黒の短毛", "白の短毛", "茶色のぶち", "グレー一色", "クリームの短毛", "その他（直接入力）"] {allow-input: true}
二匹目の体 = "小柄" #@param ["小柄", "普通", "がっしり", "脚が長め", "その他（直接入力）"] {allow-input: true}
二匹目の衣装 = "何も着けない" #@param ["無地の首輪", "ニット", "何も着けない", "バンダナ", "その他（直接入力）"] {allow-input: true}

#@markdown 最後に少し出る人
人の性別 = "女性" #@param ["女性", "男性", "その他（直接入力）"] {allow-input: true}
人の年代 = "40代" #@param ["20代", "30代", "40代", "50代", "60代", "その他（直接入力）"] {allow-input: true}
人の髪型 = "ショート" #@param ["ショート", "ボブ", "ロング", "結び", "その他（直接入力）"] {allow-input: true}
人の髪色 = "黒" #@param ["黒", "暗めの茶", "明るめの茶", "グレー", "その他（直接入力）"] {allow-input: true}
人の服 = "無地のニット" #@param ["無地のニット", "シャツ", "ジャケット", "ワンピース", "その他（直接入力）"] {allow-input: true}
人の体 = "普通" #@param ["細身", "普通", "がっしり", "背が高め", "その他（直接入力）"] {allow-input: true}
人の表情 = "ほぼ無し" #@param ["ほぼ無し", "ナチュラル", "はっきりした眉", "やわらかい笑顔", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "リビング" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "北欧" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "白と木" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "昼の自然光" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調（2匹共通）
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "junjun_ranran")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "一匹目の種類": 一匹目の種類,
            "一匹目の猫種": 一匹目の猫種,
            "一匹目の毛": 一匹目の毛,
            "一匹目の体": 一匹目の体,
            "一匹目の衣装": 一匹目の衣装,
            "二匹目の種類": 二匹目の種類,
            "二匹目の猫種": 二匹目の猫種,
            "二匹目の毛": 二匹目の毛,
            "二匹目の体": 二匹目の体,
            "二匹目の衣装": 二匹目の衣装,
            "人の性別": 人の性別,
            "人の年代": 人の年代,
            "人の髪型": 人の髪型,
            "人の髪色": 人の髪色,
            "人の服": 人の服,
            "人の体": 人の体,
            "人の表情": 人の表情,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


In [ ]:
#@title 婚活：男女の見た目 { display-mode: "form" }
#@markdown 成人の男女が会話して、最後に関係が変わる。60秒。初期値は成人の2人です。元のアカウントの人ではありません。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 女性
女性の性別 = "女性" #@param ["女性", "男性", "その他（直接入力）"] {allow-input: true}
女性の年代 = "30代" #@param ["20代", "30代", "40代", "50代", "60代", "その他（直接入力）"] {allow-input: true}
女性の髪型 = "ショート" #@param ["ショート", "ボブ", "ロング", "結び", "その他（直接入力）"] {allow-input: true}
女性の髪色 = "黒" #@param ["黒", "暗めの茶", "明るめの茶", "グレー", "その他（直接入力）"] {allow-input: true}
女性の服 = "無地のニット" #@param ["無地のニット", "シャツ", "ジャケット", "ワンピース", "その他（直接入力）"] {allow-input: true}
女性の体 = "普通" #@param ["細身", "普通", "がっしり", "背が高め", "その他（直接入力）"] {allow-input: true}
女性の表情 = "ほぼ無し" #@param ["ほぼ無し", "ナチュラル", "はっきりした眉", "やわらかい笑顔", "その他（直接入力）"] {allow-input: true}

#@markdown 男性
男性の性別 = "男性" #@param ["女性", "男性", "その他（直接入力）"] {allow-input: true}
男性の年代 = "30代" #@param ["20代", "30代", "40代", "50代", "60代", "その他（直接入力）"] {allow-input: true}
男性の髪型 = "ショート" #@param ["ショート", "ボブ", "ロング", "結び", "その他（直接入力）"] {allow-input: true}
男性の髪色 = "黒" #@param ["黒", "暗めの茶", "明るめの茶", "グレー", "その他（直接入力）"] {allow-input: true}
男性の服 = "シャツ" #@param ["無地のニット", "シャツ", "ジャケット", "ワンピース", "その他（直接入力）"] {allow-input: true}
男性の体 = "普通" #@param ["細身", "普通", "がっしり", "背が高め", "その他（直接入力）"] {allow-input: true}
男性の表情 = "ほぼ無し" #@param ["ほぼ無し", "ナチュラル", "はっきりした眉", "やわらかい笑顔", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "カフェ" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "モダン" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "生成り" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "暖かい室内灯" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "yako.shiawasekon")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "女性の性別": 女性の性別,
            "女性の年代": 女性の年代,
            "女性の髪型": 女性の髪型,
            "女性の髪色": 女性の髪色,
            "女性の服": 女性の服,
            "女性の体": 女性の体,
            "女性の表情": 女性の表情,
            "男性の性別": 男性の性別,
            "男性の年代": 男性の年代,
            "男性の髪型": 男性の髪型,
            "男性の髪色": 男性の髪色,
            "男性の服": 男性の服,
            "男性の体": 男性の体,
            "男性の表情": 男性の表情,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


# 実行

「選ぶ」のあと、このセルだけ実行します。

- 表を見る … 4ジャンルの表。ジョブは書きません。動画にもなりません
- 話でジョブを書く … 選んだ1件のジョブ。静止画が空なら T2V。ファイルがあればその画像が最初のコマ
- 自分の文で1本 … 手入力。静止画が空なら T2V、ファイルがあれば I2V。ここでは焼きません
- 元動画を再現 … 元の mp4 を Ref2VA の参照にする。カットと無音の位置で 5〜14.4 秒の範囲に分ける。読めたせりふは口がその文だけを作る。FL2VA の Turbo は載せない。ここでは焼きません


In [ ]:
#@title 実行 { display-mode: "form" }
#@markdown 「選ぶ」のあと、このセルを実行する。表なら表。話ならジョブ。自分の文なら手入力。元動画なら Ref2VA の範囲。静止画が空なら、話と自分の文は T2V。
#@markdown このセルは焼かない。ready と出たら、一番下の「焼く」を押す。投稿しない。

import os
import shutil
import subprocess
import sys
import affi_bake
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

if "何をする" not in globals():
    print("先に上の「選ぶ」を実行してください。")
else:
    out = Path("/content/affi-bake") if Path("/content").is_dir() else Path("affi-bake")
    if 何をする == "元動画を再現" and Path("/content").is_dir():
        if shutil.which("ffprobe") is None:
            subprocess.check_call(["apt-get", "update", "-qq"])
            subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])
        try:
            import faster_whisper
        except ImportError:
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"])
            except Exception as exc:
                print("せりふの読み取りは入れられなかった。口は参照の音声に合わせる。", exc)
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN") or ""
        except Exception:
            token = ""
        if token:
            os.environ["HF_TOKEN"] = token
            print("HF_TOKEN を読みました（値は表示しません）")
            for line in affi_bake.affi_media.speaker_gate_lines():
                print(line)
            if not affi_bake.affi_media.speaker_site_ready():
                print("話者を分ける pyannote を別のフォルダに入れます。H3 の torch は変えません。")
                try:
                    subprocess.check_call(affi_bake.affi_media.speaker_install_argv())
                except Exception as exc:
                    print("pyannote は入れられなかった。話者は1人として並べる。", exc)
        else:
            print("HF_TOKEN が無いので話者は分けない。")
    text = affi_bake.run_choice(
        何をする,
        handle=globals().get("HANDLE"),
        mode=globals().get("MODE"),
        fill_label=globals().get("中身", "テンプレ"),
        lines_text=globals().get("台詞", ""),
        image=globals().get("静止画", ""),
        look=globals().get("look"),
        prompt=globals().get("プロンプト", ""),
        video=globals().get("元動画", ""),
        duration_s=globals().get("秒", "10"),
        aspect=globals().get("画面", "9:16"),
        bake_here=False,
        data_dir=globals().get("DATA_DIR"),
        out_dir=out,
        table_out=globals().get("TABLE_OUT"),
        strong_min=int(globals().get("STRONG_MIN", 10)),
        compare_min=int(globals().get("COMPARE_MIN", 3)),
        thin_below=int(globals().get("THIN_BELOW", 10)),
        min_gap_pt=int(globals().get("MIN_GAP_PT", 10)),
        split=globals().get("範囲の切り方"),
        title=globals().get("題字", ""),
        logo=globals().get("ロゴ", ""),
        bgm=globals().get("曲", ""),
    )
    if 何をする == "表を見る" and display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


# 焼く

別のノートは開かない。最初にランタイムを GPU にする。途中で変えない。

上の「実行」が ready になってから、焼くにチェックを入れてこのセルを押す。

残っている範囲を続けて焼きます。全部そろったら、その場で1本につなぎます。1本だけ見たいときは「最初の1本だけ」をオンにします。

焼いた mp4 は、Colab のディスクに書いてから、そのときマウントしたアカウントのマイドライブ `affi-bake` にコピーします。
マイドライブへ直接 mp4 を開くと、小さな json だけ残って終了コード 1 になります。
json と同じフォルダの `clips` に mp4 が入ります。json だけなら動画はまだ出ていません。
失敗したときは、その場に理由が出て、`clips` に `.error.txt` が残ります。`request.json` は動画ではありません。
ランタイムを切っても、コピーした mp4 は残ります。

重みはマイドライブの `h3-weights` です。
無いときだけ「重みが無いとき落とす」を入れます。話と手入力のプレビューは 144.1GB です。オフなら落としません。
元動画の再現は `transformer_ref` を使います。FL2VA/ と Ref2VA/ の単一ファイルは落としません。
全部の範囲の mp4 がそろったとき、音は PCM のままつなぎ（`story.mov` / `source.mov`）、mp4（`story.mp4` / `source.mp4`）を最後に1回だけ作ります。
同じ内容で焼いた範囲は焼き直しません。ランタイムが切れても、マイドライブに同じ内容の範囲があれば戻します。
字幕を焼く話は、日本語のフォント（fonts-noto-cjk）を入れてから焼きます。
元動画の再現は、範囲ごとに `clips/<番号>.match.json` に、音のずれ、カット、せりふの文字の違いを書きます。口の動きそのものは測りません。

投稿しません。


In [ ]:
#@title 焼く { display-mode: "form" }
#@markdown ready のあと、焼くにチェックを入れてこのセルを押す。別のノートは開かない。投稿しない。
#@markdown チェックを入れると、残っている範囲を続けて焼いて、全部そろったら1本につなぐ。1本だけ見たいときは「最初の1本だけ」をオン。
#@markdown 同じ内容で焼いた範囲は焼き直さない。ランタイムが切れても、マイドライブに同じ内容があれば戻して続きから焼く。
#@markdown 全部の範囲がそろったら、音は PCM のままつなぎ、mp4 を最後に1回だけ作る。元動画の再現は、範囲ごとに音のずれ、カット、せりふを測って出す。
#@markdown 重みが無いとき落とすは、マイドライブに MiniMax-H3 が無いときだけ。プレビューは 144.1GB。オフなら落とさない。
焼く = False #@param {type:"boolean"}
最初の1本だけ = False #@param {type:"boolean"}
重みが無いとき落とす = False #@param {type:"boolean"}

import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import affi_bake

if not 焼く:
    print("焼くにチェックを入れて、このセルをもう一度押してください。今は焼きません。")
else:
    task = globals().get("何をする")
    if not task:
        print("先に「選ぶ」と「実行」を押してください。")
    elif affi_bake.resolve_task(task) == "table":
        print("表は動画になりません。話か自分の文を選んで、「実行」のあと、もう一度ここを押してください。")
    else:
        try:
            import torch
        except ImportError:
            torch = None
        if torch is None or not torch.cuda.is_available():
            print("GPU がオフです。ランタイムを GPU にして、読み込みからやり直してください。途中で切り替えると、ここまでのファイルが消えます。")
        else:
            props = torch.cuda.get_device_properties(0)
            vram = props.total_memory / 1024 ** 3
            print(f"GPU: {props.name}  VRAM: {vram:.1f} GB")
            if vram < 24:
                print("VRAM が 24GB 未満です。G4 に変えて、読み込みからやり直してください。")
            else:
                on_colab = Path("/content").is_dir()
                if on_colab:
                    from google.colab import drive, userdata

                    drive.mount("/content/drive")
                    print("今マウントしたアカウントのマイドライブに保存します。")
                    root = Path("/content/Research")
                    branch = "cursor/affi-template-bake-44d6"
                    repo = "https://github.com/fireworker011/Research.git"
                    script = root / "h3-runner" / "run_h3.py"
                    if not script.is_file():
                        if root.exists():
                            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
                        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
                    else:
                        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
                        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
                        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
                    sys.path.insert(0, str(root / "h3-runner"))
                    sys.path.insert(0, str(root / "research" / "affi-templates"))
                    import importlib

                    importlib.reload(affi_bake)
                else:
                    userdata = None
                    found = affi_bake._runner_root()
                    if found is None:
                        raise SystemExit("h3-runner が無い。Colab の GPU でこのセルを押してください。")
                    root = found
                cache = Path("/content/drive/MyDrive/h3-weights") if on_colab else Path("hf-cache")
                os.environ["H3_HF_CACHE"] = str(cache)
                if shutil.which("ffmpeg") is None:
                    subprocess.check_call(["apt-get", "update", "-qq"])
                    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])
                need = False
                try:
                    import av
                    import diffusers
                    import imageio
                    import soundfile
                    from torchao.quantization import FqnToConfig
                except Exception:
                    need = True
                if need:
                    pkgs = [
                        "git+https://github.com/huggingface/diffusers.git@5ff8e59ff9fe81c6e2df4fb4c6ea0d97a5df5ab2",
                        "transformers>=4.45.0",
                        "accelerate>=0.34.0",
                        "safetensors>=0.4.3",
                        "huggingface_hub>=0.25.0",
                        "av>=11.0.0",
                        "imageio>=2.34.0",
                        "imageio-ffmpeg>=0.5.0",
                        "soundfile>=0.12.0",
                    ]
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs])
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torchao==0.18.0"])
                    from torchao.quantization import FqnToConfig
                print("パッケージは揃っています", "ffmpeg", shutil.which("ffmpeg"))
                missing = affi_bake.missing_weight_files(cache, task)
                if missing and not 重みが無いとき落とす:
                    print("重みが無いので焼きません。落とすときは「重みが無いとき落とす」を入れて、もう一度押してください。")
                    for item in missing:
                        print(item)
                else:
                    path = affi_bake.commands_file(
                        task,
                        handle=globals().get("HANDLE"),
                        mode=globals().get("MODE"),
                        out_dir=Path("/content/affi-bake") if on_colab else Path("affi-bake"),
                    )
                    if path is None:
                        print("先に「選ぶ」を押してください。")
                    else:
                        if on_colab:
                            back = affi_bake.restore_done_clips(path)
                            if back:
                                print("マイドライブから、同じ内容で焼いてある範囲を戻した", len(back), "本")
                        lines, note = affi_bake.commands_to_run(path, first_only=bool(最初の1本だけ), skip_done=True)
                        print(note)
                        if lines and missing:
                            token = ""
                            if userdata is not None:
                                try:
                                    token = userdata.get("HF_TOKEN") or ""
                                except Exception:
                                    token = ""
                            if token:
                                os.environ["HF_TOKEN"] = token
                                os.environ["HUGGING_FACE_HUB_TOKEN"] = token
                                print("HF_TOKEN を読みました（値は表示しません）")
                            prep = affi_bake.prepare_command(lines[0], cache)
                            code = affi_bake.run_logged(shlex.split(prep), root)
                            print(f"重みの終了コード {code}")
                            if code != 0:
                                print("失敗。重みは揃っていません。mp4 は出来ていません。")
                                lines = []
                            else:
                                sys.path.insert(0, str(root / "h3-runner"))
                                if affi_bake.resolve_task(task) != "repro":
                                    from h3_runner.loras import prepare_fast_loras

                                    prepare_fast_loras(cache / "loras")
                        repro = affi_bake.resolve_task(task) == "repro"
                        made = False
                        finished = True
                        for line in lines:
                            print(line)
                            code = affi_bake.run_logged(shlex.split(line), root)
                            print(f"終了コード {code}")
                            local_out = affi_bake._out_of(line)
                            if code != 0:
                                print("失敗。mp4 は出来ていません。")
                                if local_out is not None:
                                    err = local_out.with_suffix(".error.txt")
                                    if err.is_file():
                                        print(err.read_text(encoding="utf-8"))
                                finished = False
                                break
                            if local_out is None:
                                continue
                            if not local_out.is_file():
                                request = local_out.with_suffix(".request.json")
                                if request.is_file():
                                    print("mp4 は出ていない。あるのは", request)
                                else:
                                    print("mp4 は出ていない。", local_out)
                                finished = False
                                continue
                            print("書いた", local_out, local_out.stat().st_size, "bytes")
                            affi_bake.mark_done(line)
                            made = True
                            if repro:
                                try:
                                    print(affi_bake.measure_repro_clip(path.parent, local_out))
                                except Exception as exc:
                                    print("一致は測れなかった", exc)
                            if on_colab:
                                try:
                                    saved = affi_bake.publish_job_dir(path.parent)
                                except Exception as exc:
                                    print("マイドライブへのコピーに失敗した", exc)
                                else:
                                    print("マイドライブにコピーした", saved)
                        if finished and affi_bake.all_clips_made(path.parent):
                            if on_colab and affi_bake.needs_caption_font(path.parent):
                                subprocess.check_call(["apt-get", "update", "-qq"])
                                subprocess.check_call(["apt-get", "install", "-y", "-qq", affi_bake.affi_finish.FONT_PACKAGE])
                            joined = True
                            for join in affi_bake.finish_lines(path.parent):
                                print(join)
                                code = affi_bake.run_logged(shlex.split(join), root)
                                print(f"つなぎの終了コード {code}")
                                if code != 0:
                                    joined = False
                                    break
                            if joined and on_colab:
                                try:
                                    saved = affi_bake.publish_job_dir(path.parent)
                                except Exception as exc:
                                    print("マイドライブへのコピーに失敗した", exc)
                                else:
                                    for item in sorted(saved.glob("*.mp4")):
                                        print("つないだ mp4", item, item.stat().st_size, "bytes")
                        elif bool(最初の1本だけ) and made:
                            print("最初の1本だけ焼いた。全部つなぐときは、最初の1本だけをオフにして、もう一度押す。")
                        elif made:
                            print("途中で止まった。焼いてある範囲は飛ばして、もう一度押すと続きから焼く。")
                        if made:
                            print("投稿していない。")


# 投稿したあとの数字

今は飛ばします。

`research/affi-templates/reference-accounts/results.csv` に1行足す。空欄は0にしない。未入力のままにする。

- `hypothesis_id` … H1-1 のような番号
- `variant` … A か B
- `views` … 72時間後の再生
- `two_sec_hold_pct` … 2秒視聴のパーセント
- `diagnosis_signups` … 婚活の診断申込。無い仮説は空欄

本数が足りないと「本数不足」。基準を超えると A か B が出る。A/Bで変えるのは仮説の1項目だけ。見た目は両版で同じにする。
